# Distributor RFQ → Item Matching Prototype (v2)

Extended version with 13 real RFQ examples and 6 match types:

| Match Type | Meaning |
|---|---|
| 🟢 **Exact** | Direct hit on Item Number — ready to quote |
| 🔵 **Cross Ref** | OEM part mapped to DST equivalent (per Estimate #10001) |
| 🟡 **Cross Ref Needed** | OEM part with no DST mapping yet — needs lookup |
| 🟠 **Partial** | Similar catalog item(s) found, needs review |
| 🟣 **Description Match** | No PN given, but description keywords matched catalog items |
| ⚫ **Out of Scope** | X-Ray / non-NDT product (RT-* / RS-*) |
| 🔴 **No Match** | Nothing reasonable found — manual review |

**How to run:** Run cells top-to-bottom. When prompted in Cell 2, upload `IMS_Items_Exp.csv`.

## 1. Imports & Engine

In [ ]:
"""RFQ -> Item Matching prototype. Distributor demo (v2 - extended)."""
import re
import pandas as pd
from difflib import get_close_matches

# --- Hardcoded OEM->DST cross-ref (from Distributor Estimate #10001) ---
OEM_TO_DST = {
    "OEM9087L": "DST-0615L",
    "OEM9087R": "DST-0615R",
    "OEM9110L": "DST-0616L",
    "OEM9110R": "DST-0616R",
}

# Out-of-scope product prefixes (X-Ray, radiation signs - not Distributor NDT line)
OUT_OF_SCOPE_PREFIXES = ("RT-", "RS-")

# --- Normalization ---
def normalize(s):
    if s is None:
        return ""
    return re.sub(r"[\s\-_/.,:]+", "", str(s)).upper()

# --- Noise vocabulary ---
NOISE = {
    "PN", "QTY", "EA", "UM", "UOM", "CONDITION", "DESCRIPTION", "DESC",
    "PART", "NUMBER", "PARTNUMBER", "REMARKS", "ITEM", "BIDS", "BID",
    "RFQ", "PLEASE", "EQUIVALENT", "FREQUENCY", "DIAMETER", "WORKING",
    "LENGTH", "TIP", "COIL", "DISTANCE", "MM", "KHZ", "MHZ", "TO", "OF",
    "C", "TDS", "TEST", "REPORT", "FLEXIBLE", "ROTO", "PROBE", "SUBMIT",
    "FOR", "FROM", "AND", "THE", "WITH", "BLOCK", "REFERENCE", "STANDARD",
    "BNC", "UHF", "CABLE", "ULTRASONIC", "TRANSDUCER", "CURRENCY",
    "UNIT", "PRICE", "LEAD", "TIME", "CERTIFICATE", "USML", "ECCN",
    "ALTERNATE", "MOQ", "WARRANTY", "OEM", "FAA", "EASA", "INV",
    "SHELF", "LIFE", "CURE", "DATE", "VALIDITY", "EXPORT", "LICENCE",
    "SME", "MILITARY", "SENSITIVE", "STRATEGIC", "GOODS", "ACT",
    "TREASURY", "PROGRAM", "NATIONS", "SANCTIONS", "INDICATE",
    "STATEMENT", "COMMERCIAL", "AUTHORIZED", "GOVERNMENT", "DESTINATION",
    "USER", "TRANSFERRED", "DISPOSED", "ORIGINAL", "FORM", "INCORPORATED",
    "DEPARTMENT", "STATE", "QUOTATION", "VENDOR", "ADDRESSED", "AEROSPACE",
    "SECTOR", "ENTITY", "ACCEPTED", "ANOTHER", "PROVIDE", "DOCUMENTARY",
    "PROOF", "QUOTED", "ADVISE", "SPECIFY", "CONTRACTORS", "SUPPLIERS",
    "DECLARE", "CONFLICT", "INTEREST", "COMPLY", "LEGAL", "OTHER",
    "REQUIREMENTS", "PERTAINING", "ENVIRONMENT", "HEALTH", "SAFETY",
    "ISSUES", "REGION", "CONTROL", "TREASURY", "DESIGNATED", "LIST",
    "UNITED", "COUNTRY", "ULTIMATE", "USE", "TRANSSHIPPED", "VOYAGE",
    "OTHERWISE", "ANY", "EITHER", "AFTER", "BEING", "INTO", "END",
    "ITEMS", "WITHOUT", "PRIOR", "WRITTEN", "APPROVAL",
    "MFR", "CAGE", "NSN", "APPLICABLE", "ATTACHMENT", "INFORMATION",
    "HELLO", "HELP", "MAY", "GET", "QUOTE", "ATTACHED", "ADVISE",
    "DEAR", "COULD", "SEND", "FOLLOWING",
    "TRANDUCER",
    "CLOSING",
}

UNIT_RE = re.compile(
    r"^\d+(\.\d+)?(MM|CM|IN|HZ|KHZ|MHZ|GHZ|EA|PCS|G|KG|LB|OZ|V|W|MA)"
    r"(-\d+(\.\d+)?(MM|CM|IN|HZ|KHZ|MHZ|GHZ|EA|PCS|G|KG|LB|OZ|V|W|MA))?$"
)

# Patterns to reject — these look like PNs but aren't quoteable items
JUNK_PATTERNS = [
    re.compile(r"^PR-\d{2,4}-\d+$"),          # PR-0000-00001
    re.compile(r"^DCQ\d+$"),                  # DCQ377367
    re.compile(r"^RFQ\d+$"),                  # RFQ12345
    re.compile(r"^B-?7\d{2}$"),               # B777, B-787
    re.compile(r"^A-?3\d{2}$"),               # A320, A-380
    re.compile(r"^\d{2}-\d{2}-\d{2}$"),       # 54-51-06 (NDT manual sections)
    re.compile(r"^\d{2}-\d{2}-\d{3}$"),       # 57-20-150 variant
    re.compile(r"^S/N-?\d+$"),                # S/N-358
    re.compile(r"^ASTM-?E?-?\d+$"),           # ASTM-E-1742
    re.compile(r"^\d+/\d+$"),                 # 5/16, 7/16 (fractions)
    re.compile(r"^TI-?6-?4\.?$"),             # Ti64, Ti-6-4
    re.compile(r"^AL\d{4}$"),                 # AL6061, AL2024
    re.compile(r"^\d{4}-?T\d$"),              # 2024-T3, 7075-T6
]

PN_TOKEN = r"[A-Za-z0-9][A-Za-z0-9\-_/\.]{3,}"
EQUIV_RE = re.compile(
    rf"({PN_TOKEN})\s+(?:equivalent\s+to|equivalent|equiv\.?\s*to|eq\.?\s*to)\s+({PN_TOKEN})",
    re.IGNORECASE,
)

# Description keywords for description-only RFQ matching
DESC_KEYWORDS = {
    "probe", "probet", "transducer", "kit", "reference standard",
    "calibration block", "step", "cable", "rototest", "eddy current",
    "ultrasonic", "surface wave", "shear wave",
}

def _valid_pn(u):
    if u in NOISE: return False
    if not re.search(r"\d", u): return False
    if re.fullmatch(r"\d+(\.\d+)?", u): return False
    if UNIT_RE.match(u): return False
    if len(u) < 4: return False
    for pat in JUNK_PATTERNS:
        if pat.match(u): return False
    return True

def extract_rfq_items(text):
    """Return list of {'pn', 'customer_ref'}. Detects 'X equivalent to Y' linkage."""
    primary_to_ref, skip_norm = {}, set()
    for m in EQUIV_RE.finditer(text):
        p, e = m.group(1), m.group(2)
        if not _valid_pn(p.upper().strip(".,:")) or not _valid_pn(e.upper().strip(".,:")):
            continue
        primary_to_ref[normalize(p)] = e
        skip_norm.add(normalize(e))

    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9\-_/\.]{3,}", text)
    seen, out = set(), []
    for t in tokens:
        u = t.upper().strip(".,:")
        if not _valid_pn(u): continue
        n = normalize(t)
        if n in skip_norm or n in seen: continue
        seen.add(n)
        out.append({"pn": t, "customer_ref": primary_to_ref.get(n)})
    return out

def is_out_of_scope(rfq_pn):
    """Check if part number prefix indicates out-of-scope product line."""
    upper = rfq_pn.upper().strip()
    return any(upper.startswith(pfx) for pfx in OUT_OF_SCOPE_PREFIXES)

def description_match(text, catalog_df, top_n=3):
    """When no PNs found, search description for keyword hits in catalog."""
    text_lower = text.lower()
    found_keywords = [kw for kw in DESC_KEYWORDS if kw in text_lower]
    if not found_keywords:
        return []
    # rank catalog rows by how many keywords appear in their Item Name
    scores = []
    for i, row in catalog_df.iterrows():
        name = str(row.get("Item Name", "")).lower()
        score = sum(1 for kw in found_keywords if kw in name)
        if score > 0:
            scores.append((score, i))
    scores.sort(reverse=True)
    return [(catalog_df.iloc[i], found_keywords) for _, i in scores[:top_n]]

# --- Matching engine ---
def match_one(rfq_pn, catalog_df, norm_index, norm_list):
    n = normalize(rfq_pn)

    # Step 0: Out-of-scope check (X-Ray, etc.)
    if is_out_of_scope(rfq_pn):
        return {
            "Match Type": "Out of Scope",
            "Internal Part Number": "",
            "Confidence": "Low",
            "Notes": "X-Ray / non-NDT product - outside Distributor scope",
        }

    # Step 1: Exact
    if n in norm_index:
        row = catalog_df.iloc[norm_index[n]]
        return {
            "Match Type": "Exact",
            "Internal Part Number": row["Item Number"],
            "Confidence": "High",
            "Notes": f"Exact catalog match ({row['Item Name']})",
        }

    # Step 2: Cross Ref (hardcoded OEM->DST map)
    if n in OEM_TO_DST:
        return {
            "Match Type": "Cross Ref",
            "Internal Part Number": OEM_TO_DST[n],
            "Confidence": "High",
            "Notes": "Mapped from OEM to DST (per Estimate #10001)",
        }

    # Step 2b: Generic OEM pattern (no specific mapping yet)
    if n.startswith("OEM"):
        return {
            "Match Type": "Cross Ref Needed",
            "Internal Part Number": "",
            "Confidence": "Low",
            "Notes": "OEM competitor part - DST equivalent not yet mapped, needs lookup",
        }

    # Step 3: Partial
    candidates = get_close_matches(n, norm_list, n=2, cutoff=0.6)
    if not candidates and len(n) >= 4:
        candidates = [x for x in norm_list if n in x or x in n][:2]
    if candidates:
        picks = [f"{catalog_df.iloc[norm_index[c]]['Item Number']} ({catalog_df.iloc[norm_index[c]]['Item Name']})" for c in candidates]
        return {
            "Match Type": "Partial",
            "Internal Part Number": "; ".join(catalog_df.iloc[norm_index[c]]["Item Number"] for c in candidates),
            "Confidence": "Medium",
            "Notes": "Similar item, needs review: " + " | ".join(picks),
        }

    # Step 4: No Match
    return {
        "Match Type": "No Match",
        "Internal Part Number": "",
        "Confidence": "Low",
        "Notes": "Not found in catalog - manual review required",
    }

# --- Pipeline ---
def run_pipeline(rfq_texts, catalog_path):
    cat = pd.read_csv(catalog_path, skiprows=2)
    cat = cat.dropna(subset=["Item Number"]).reset_index(drop=True)
    norm_index = {}
    for i, pn in enumerate(cat["Item Number"]):
        n = normalize(pn)
        if n and n not in norm_index:
            norm_index[n] = i
    norm_list = list(norm_index.keys())

    rows = []
    for label, text in rfq_texts.items():
        items = extract_rfq_items(text)

        if not items:
            # Description-only RFQ (Ex 4, 8)
            desc_hits = description_match(text, cat)
            if desc_hits:
                picks = [f"{r['Item Number']} ({r['Item Name']})" for r, _ in desc_hits]
                kws = desc_hits[0][1]
                rows.append({
                    "Source RFQ": label,
                    "RFQ Item": "(description-only - no PN given)",
                    "Customer Reference": "",
                    "Match Type": "Description Match",
                    "Internal Part Number": "; ".join(r["Item Number"] for r, _ in desc_hits),
                    "Confidence": "Medium",
                    "Notes": f"Keyword match on [{', '.join(kws)}]: " + " | ".join(picks),
                })
            else:
                rows.append({
                    "Source RFQ": label,
                    "RFQ Item": "(no part number or recognizable spec)",
                    "Customer Reference": "",
                    "Match Type": "No Match",
                    "Internal Part Number": "",
                    "Confidence": "Low",
                    "Notes": "Custom build request - engineering review required",
                })
            continue

        for item in items:
            pn, cref = item["pn"], item["customer_ref"]
            r = match_one(pn, cat, norm_index, norm_list)
            if cref:
                r["Notes"] = f"{r['Notes']}; customer also referenced equivalent '{cref}'"
            rows.append({"Source RFQ": label, "RFQ Item": pn,
                         "Customer Reference": cref or "", **r})

    return pd.DataFrame(rows, columns=["Source RFQ", "RFQ Item", "Customer Reference",
                                        "Match Type", "Internal Part Number",
                                        "Confidence", "Notes"])

## 2. Upload the catalog CSV
When the upload widget appears, select **`IMS_Items_Exp.csv`**.

In [ ]:
from google.colab import files
uploaded = files.upload()
CATALOG_PATH = list(uploaded.keys())[0]
print(f'Loaded: {CATALOG_PATH}')

## 3. The 13 RFQ examples
4 original examples + 9 new examples from post-demo feedback. Edit any of these to test new RFQs.

In [ ]:
# --- All 13 RFQs (4 original + 9 new) ---
RFQS = {
    "RFQ 1": """Can you please help me out with a sales quote on the following:
PN: 2-164-551 BLOCK, REFERENCE STANDARD
QTY: 1""",

    "RFQ 2": """Please submit your Bids for the items listed below:
PartNumber Description Condition Qty/UM
SC-BNC-UHF-6  BNC to UHF CABLE FOR ULTRASONIC TRANSDUCER  2 EA""",

    "RFQ 3": """Flexible roto probe
DS-5007/7.8MM equivalent to KD1H52408-2
QTY: 1
Specs: Frequency 100KHZ-2MHZ, Diameter 7.8 mm, working length 90mm

Flexible roto probe
DS-5007/6.1MM equivalent to KD1H52404-2
QTY: 1
Specs: Frequency 100KHZ-2MHZ, Diameter 6.1 mm, working length 90mm""",

    "RFQ 4": """Please quote the following PN:
NDT3317
NDT3318
NDT3310-A
NDT3310-B
NDT3310-C
OEM-9087L
OEM-9087-R
OEM-9110L
OEM-9110R""",

    "RFQ 5 (Ex1 X-Ray mix)": """Description Part No Quantity Note
Package of 26 Letters A Through Z, Unmounted Ray-Check Lead Letters | RT-RC-LL-U26AZ-Master | 1 | X-RAY processing equipment
Package of 10 Single Number, Unmounted Ray-Check Lead Letters | RT-RC-LL-U10N-Master | 1
NDS Products Plastic Radiation Area Sign Caution X-Ray | RS-NDS-PL4 | 3
X-Rite 331C Portable Transmission Densitometer | RT-XTDM-331C | 1
Kowotest BAM 15 Step Density Tablet | RT-KT-13-40205 | 1
ASTM-E-1742 IQI Sets Plaque Type | RT-RC-PK-ASTM-E-1742-A-Master | 1""",

    "RFQ 6 (Ex2 PR-0000-00001)": """PR-0000-00001
ITEM 1: DST-1371 ULTRASONIC TRANSDUCER 5MHZ 45 DEG SHEAR WAVE IN TITANIUM. QTY 1 EA
ITEM 2: NDT777A NEW ULTRASONIC REFERENCE STANDARD PER B777 PART 4 54-51-06 FIG 2. QTY 1 EA
Closing Date: 07 May 2026""",

    "RFQ 7 (Ex3 DCQ377367)": """RFQ Number: DCQ377367
Item 12: AUR.875SE/3.0WL  PROBE PROBE BOLT HO  Qty 2  NE
Item 17: 8382199  BUMPER  Qty 11  NE""",

    "RFQ 8 (Ex4 custom blocks)": """Requesting a quote for 10 total calibration blocks all provided with a Delrin holding block, test report, and certificate of conformance.
5 of 8 steps, thicknesses 0.020", 0.025", 0.030", 0.040", 0.100", 0.200", 0.300", 0.500", 1"x1" pad, +/- 0.001" tolerance, 304 Stainless Steel.
5 of 8 steps, same thicknesses and pad, Ti64.
For reference, similar to DSNDT-3406AL-1 from previous Quote #37733.""",

    "RFQ 9 (Ex5 NDT list)": """Could you send a quote for following PN?
NDT6109 (REFERENCE STANDARD NDT) QTY 2
NDT666A (REFERENCE STANDARD NDT) QTY 1
NDT666B (REFERENCE STANDARD NDT) QTY 1
NDT666C (REFERENCE STANDARD NDT) QTY 1""",

    "RFQ 10 (Ex6 NDT6131)": """Our NDI partners at Boeing designed a new reference standard NDT6131 for medium frequency eddy current (MFEC) inspection. Please provide quote for 1 standard and 3 probes.""",

    "RFQ 11 (Ex7 mixed PNs)": """Can you please assist us with a quote?
1: SUS921L TRANSDUCER QTY 2
2: SUS921R TRANSDUCER QTY 2
3: NEC1062 PROBE: S/N-358 QTY 2
4: NEC1062 PROBE: S/N-359 QTY 2
5: 291-719-800 TRANSDUCER QTY 2
6: SUSM570AT TRANSDUCER QTY 2
7: A109S-RM TRANSDUCER QTY 2""",

    "RFQ 12 (Ex8 rototest kit)": """I need a rototest kit for eddy current inspection with 1/4", 5/16", and 7/16" Probets, or Kit de probets 1/4, 5/16 and 7/16""",

    "RFQ 13 (Ex9 mixed list)": """1: NDT777A Reference standard QTY 1 (B777 NDT PART 4, SUBJECT 54-51-06)
2: DST-1371 Ultrasonic transducer QTY 2
3: NDT771 Reference standard QTY 1 (B777 NDT PART 4, SUBJECT 57-10-05)
4: NDT774 Reference standard QTY 1
5: OEM-5127L-I Ultrasonic transducer QTY 1
6: OEM-5127L-O Ultrasonic transducer QTY 1
7: OEM-5127R-I Ultrasonic transducer QTY 1
8: OEM-5127R-O Ultrasonic transducer QTY 1
9: NDT6130-1 Reference standard QTY 1 (B777 NDT PART 4, SUBJECT 57-20-17)
10: NDT6130-2 Reference standard QTY 1
11: NDT6130-3 Reference standard QTY 1""",
}

## 4. Run the pipeline

In [ ]:
result = run_pipeline(RFQS, CATALOG_PATH)
print(f'Total rows: {len(result)}\n')

def _color(row):
    c = {'Exact': '#d4edda', 'Cross Ref': '#cce5ff',
         'Cross Ref Needed': '#ffeaa7', 'Partial': '#fff3cd',
         'Description Match': '#e2d4f0', 'Out of Scope': '#e2e3e5',
         'No Match': '#f8d7da'}.get(row['Match Type'], '')
    return [f'background-color: {c}'] * len(row)

result.style.apply(_color, axis=1)

## 5. Summary counts

In [ ]:
summary = result['Match Type'].value_counts().to_frame('Count')
summary['%'] = (summary['Count'] / len(result) * 100).round(1)
print(summary.to_string())
print(f'\nTotal RFQ items processed: {len(result)}')
print(f'Across {len(RFQS)} distinct RFQs')

## 6. Download CSV

In [ ]:
OUT_PATH = 'rfq_match_output.csv'
result.to_csv(OUT_PATH, index=False)
from google.colab import files
files.download(OUT_PATH)